In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

In [4]:
!pip install icu-sepsis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 18.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s  0:00:00
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827727 sha256=6d66b5c601534f1c37015225c6b3e4792b7028d7adf4614d1d0300006c9a307a
  Stored in directory: /Users/anamacedo/Library/Caches/pip/wheels/1d/34/c6/856a1e1eff47d8f18545c833b6138ae1e9f53c7de9bcc5f31d
Successfully built gym
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [icu-sepsis]


In [10]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from envs.env_setup import make_sepsis_env
from utils.evaluation import eval_agent, print_results
from utils.plotting import plot_comparison

In [11]:
def compute_expected_reward(P, R):
    """
    Converte a matriz de recompensa 3D R(s,a,s') para R_expected(s,a).

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)

    Returns:
        R_expected: matriz com shape (n_states, n_actions)
    """
    # Para cada (s,a), somamos P(s,a,s') * R(s,a,s') sobre todos os s'
    R_expected = np.sum(P * R, axis=2)
    return R_expected

In [12]:
def compute_value_function(policy, P, R_expected, gamma=1.0):
    """
    Calcula a função de valor para uma política fixa (Policy Evaluation).

    Args:
        policy: array com shape (n_states,) — acção a tomar em cada estado
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R_expected: matriz de recompensas com shape (n_states, n_actions)
        gamma: fator de desconto

    Returns:
        value_table: array com shape (n_states,) — valor de cada estado
    """
    # número de iterações máximo
    num_iterations = 10000

    # threshold para convergência (igual às aulas)
    threshold = 1e-6

    n_states = P.shape[0]

    # inicializamos a value table com zeros (igual às aulas)
    value_table = np.zeros(n_states)

    # para cada iteração
    for i in range(num_iterations):

        # guardamos uma cópia da value table anterior
        updated_value_table = np.copy(value_table)

        # para cada estado
        for s in range(n_states):

            # seleccionamos a acção que a política indica para este estado
            a = int(policy[s])

            # calculamos o valor do estado usando a equação de Bellman:
            # V(s) = sum_{s'} P(s'|s,a) * [R(s,a) + gamma * V(s')]
            value_table[s] = sum([
                P[s, a, s_next] * (R_expected[s, a] + gamma * updated_value_table[s_next])
                for s_next in range(n_states)
            ])

        # verificamos se a diferença entre a value table atual e a anterior
        # é menor que o threshold — se sim, convergiu e paramos
        if np.max(np.abs(updated_value_table - value_table)) <= threshold:
            break

    return value_table

In [13]:
def extract_policy(value_table, P, R_expected, gamma=1.0):
    """
    Extrai a política óptima a partir da função de valor (Policy Improvement).

    Args:
        value_table: array com shape (n_states,) — função de valor actual
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R_expected: matriz de recompensas com shape (n_states, n_actions)
        gamma: fator de desconto

    Returns:
        policy: array com shape (n_states,) — acção óptima em cada estado
    """
    n_states, n_actions, _ = P.shape

    # inicializamos a política com zeros
    policy = np.zeros(n_states)

    # para cada estado
    for s in range(n_states):

        # calculamos o Q-value para cada acção possível
        Q_values = [
            sum([
                P[s, a, s_next] * (R_expected[s, a] + gamma * value_table[s_next])
                for s_next in range(n_states)
            ])
            for a in range(n_actions)
        ]

        # escolhemos a acção com maior Q-value (passo greedy)
        policy[s] = np.argmax(np.array(Q_values))

    return policy

In [14]:
def policy_iteration(P, R, gamma=1.0):
    """
    Policy Iteration num MDP tabular.

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)
        gamma: fator de desconto (1.0 por omissão — sem desconto temporal,
               igual à convenção do ambiente ICU-Sepsis)

    Returns:
        policy: array com shape (n_states,) — política óptima
        value_function: array com shape (n_states,) — função de valor óptima
        convergence_deltas: lista com o delta máximo por iteração (para gráfico)
    """
    n_states = P.shape[0]

    # calculamos o R_expected (s,a) a partir do R(s,a,s') do ambiente
    R_expected = compute_expected_reward(P, R)

    # começamos com uma política inicial aleatória (igual às aulas)
    policy = np.random.randint(0, P.shape[1], n_states)

    # inicializamos a value function com zeros
    value_function = np.zeros(n_states)

    # número máximo de iterações
    num_iterations = 1000

    # lista para guardar os deltas de convergência (para o gráfico do relatório)
    convergence_deltas = []

    # para cada iteração
    for i in range(num_iterations):

        # --- PASSO 1: Policy Evaluation ---
        # calculamos a função de valor para a política actual
        value_function = compute_value_function(policy, P, R_expected, gamma)

        # --- PASSO 2: Policy Improvement ---
        # extraímos uma nova política melhorada
        new_policy = extract_policy(value_function, P, R_expected, gamma)

        # guardamos o delta para o gráfico de convergência
        delta = np.max(np.abs(value_function - np.zeros(n_states)))
        convergence_deltas.append(delta)

        # se a política não mudou, convergiu — paramos
        if np.all(policy == new_policy):
            print(f'Policy Iteration convergiu após {i + 1} iterações!')
            break

        # caso contrário, actualizamos a política
        policy = new_policy

    return policy.astype(int), value_function, convergence_deltas

In [15]:
def value_iteration(P, R, gamma=1.0):
    """
    Value Iteration num MDP tabular.

    Args:
        P: tensor de transições com shape (n_states, n_actions, n_states)
        R: tensor de recompensas com shape (n_states, n_actions, n_states)
        gamma: fator de desconto

    Returns:
        policy: array com shape (n_states,) — política óptima
        value_function: array com shape (n_states,) — função de valor óptima
        convergence_deltas: lista com o delta máximo por iteração (para gráfico)
    """
    n_states, n_actions, _ = P.shape

    # calculamos o R_expected (s,a) a partir do R(s,a,s') do ambiente
    R_expected = compute_expected_reward(P, R)

    # número máximo de iterações
    num_iterations = 10000

    # threshold para convergência (igual às aulas)
    threshold = 1e-6

    # inicializamos a value function com zeros
    value_function = np.zeros(n_states)

    # lista para guardar os deltas de convergência (para o gráfico do relatório)
    convergence_deltas = []

    # para cada iteração
    for i in range(num_iterations):

        # guardamos uma cópia da value function anterior
        updated_value_function = np.copy(value_function)

        # para cada estado
        for s in range(n_states):

            # calculamos o Q-value para cada acção possível
            Q_values = [
                sum([
                    P[s, a, s_next] * (R_expected[s, a] + gamma * updated_value_function[s_next])
                    for s_next in range(n_states)
                ])
                for a in range(n_actions)
            ]

            # actualizamos V(s) com o máximo dos Q-values (Bellman optimality)
            value_function[s] = np.max(Q_values)

        # calculamos o delta máximo entre iterações
        delta = np.max(np.abs(updated_value_function - value_function))
        convergence_deltas.append(delta)

        # se o delta é menor que o threshold, convergiu — paramos
        if delta <= threshold:
            print(f'Value Iteration convergiu após {i + 1} iterações!')
            break

    # no fim, extraímos a política óptima a partir da value function final
    policy = extract_policy(value_function, P, R_expected, gamma)

    return policy.astype(int), value_function, convergence_deltas


# ---------------------------------------------------------------------------
# get_policy — converte o array da política numa função chamável
# ---------------------------------------------------------------------------
# Necessário para usar com utils.evaluation.eval_agent()

def get_policy(policy_array):
    """
    Transforma o array da política numa função chamável.
    Necessário para a função eval_agent() de utils/evaluation.py.

    Args:
        policy_array: array com shape (n_states,)

    Returns:
        função que recebe um estado (int) e devolve uma acção (int)
    """
    return lambda obs: int(policy_array[obs])

In [16]:
# 1. Criar o ambiente e extrair P e R
env = make_sepsis_env()
raw = env.unwrapped
P = raw._tx_mat
R = raw._r_mat

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


In [17]:
# 2. Correr os algoritmos
policy_pi, V_pi, deltas_pi = policy_iteration(P, R, gamma=1.0)
policy_vi, V_vi, deltas_vi = value_iteration(P, R, gamma=1.0)

Policy Iteration convergiu após 4 iterações!
Value Iteration convergiu após 135 iterações!


In [18]:
# 3. Medir com a função oficial
from utils.evaluation import eval_agent, print_results

results_pi = eval_agent(get_policy(policy_pi), make_sepsis_env, n_eval_episodes=1000, seed=42)
results_vi = eval_agent(get_policy(policy_vi), make_sepsis_env, n_eval_episodes=1000, seed=42)

print_results("Policy Iteration", results_pi)
print_results("Value Iteration",  results_vi)

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02
make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02
=== Policy Iteration ===
  Mean return  : 0.7914 ± 0.3710
  Survival rate: 83.2%
  Mean ep len  : 10.6
=== Value Iteration ===
  Mean return  : 0.7914 ± 0.3710
  Survival rate: 83.2%
  Mean ep len  : 10.6
